# Reproducing Churchland et al. (2012) from dandiset 000070

**Paper.** Churchland MM, Cunningham JP, Kaufman MT, Foster JD, Nuyujukian P, Ryu SI, Shenoy KV
(2012). Neural population dynamics during reaching. *Nature* 487:51-56.
[doi:10.1038/nature11129](https://doi.org/10.1038/nature11129). This attempt used the full text of
the NIH author manuscript in PubMed Central (PMC3393826) and the Supplementary Information PDF from
the publisher's site.

**Dandiset.** [DANDI:000070](https://dandiarchive.org/dandiset/000070/0.260528.0919), version
0.260528.0919. The paper cites no version (it predates the dandiset). Two versions are published,
0.251218.1714 and 0.260528.0919; they hold the same ten files (same blobs and sizes, printed below)
and differ only in metadata (contributor affiliations, keywords, study target), none of which the
analysis uses. The most recent published version is pinned, as the protocol asks when the paper
cites none and no earlier version matches the paper better.

**What the dandiset holds.** Ten array sessions of monkeys Jenkins (J) and Nitschke (N) performing
the maze task, each with two 96-electrode arrays (PMd and M1). Two are the paper's array datasets:
J-array (2009-09-18) and N-array (2010-09-23). One more (`sub-Nitschke_ses-20090910`) is
consistent with the "N-9-10-2010" dataset of Supplementary Figure 5 (see the data check). The seven
single-electrode datasets, the EMG, the leech and the walking-monkey data are not in it. The units
are one per electrode (192 per session), not the paper's 146 and 218 sorted isolations.

**Code read.** The dandiset links its conversion code,
[catalystneuro/shenoy-lab-to-nwb](https://github.com/catalystneuro/shenoy-lab-to-nwb) at commit
319acb4. The paper links no code; jPCA follows the Methods, the Supplementary Derivation and the
Python port of the lab's MATLAB codepack, [bantin/jPCA](https://github.com/bantin/jPCA) at commit
3f8be9f (the lab's download page returned HTTP 403). See `reach.py` for where each parameter comes
from.

| Claim | Tested here |
|---|---|
| array-dataset-composition | yes |
| fig3ef-same-direction | yes |
| fig3-independent-of-curvature | yes |
| fig3-phase-from-prep-state | yes |
| var-captured-jpc-plane | yes |
| pc23-variance | yes |
| suppfig4-j-array-planes | yes |
| suppfig4-planes-range | yes |
| fig6a-angle-peak | yes |
| fig6b-skew-fit | yes |
| rotation-frequency | yes |
| suppfig3-shuffle-controls | yes |
| suppfig5-areas-variance | yes |
| suppfig5-m1-faster | yes |
| fig6-velocity-model | yes (simulated from the dandiset's hand kinematics) |
| fig6-complex-kinematic-model | yes (simulated from the dandiset's hand kinematics) |
| fig1-leech-and-walking | blocked: data missing |
| fig2-single-neuron-responses | blocked: data missing |
| fig3abcd-single-electrode-datasets | blocked: data missing |
| fig4-emg-no-rotation | blocked: data missing |
| fig5-generator-model | blocked: data missing |
| fig6cd-speed-independent-frequency | blocked: data missing |

In [ ]:
import time
import warnings

import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import remfile
from dandi.dandiapi import DandiAPIClient
from scipy import stats
from scipy.optimize import minimize

import reach

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_rows", 200)
warnings.filterwarnings("ignore", category=RuntimeWarning)

started = time.time()
urls = reach.asset_urls()
sessions = {}
for name, path in reach.SESSIONS.items():
    t = time.time()
    sessions[name] = reach.load_session(path, urls[path], cursor=(name == "J-array"))
    print(f"{name}: {path} loaded in {time.time() - t:.0f} s")
print(f"dandiset {reach.DANDISET_ID} version {reach.DANDISET_VERSION}")

## Data check

Each check of the metadata against the values is stated under its output.

In [ ]:
with DandiAPIClient() as client:
    ds = client.get_dandiset(reach.DANDISET_ID)
    versions = [v.identifier for v in ds.get_versions()]
    per_version = {}
    for v in versions:
        d = client.get_dandiset(reach.DANDISET_ID, v)
        per_version[v] = {a.path: (a.blob, a.size) for a in d.get_assets()}
    meta = client.get_dandiset(reach.DANDISET_ID, reach.DANDISET_VERSION).get_raw_metadata()
print("Versions:", versions)
for v in versions:
    same = per_version[v] == per_version[reach.DANDISET_VERSION]
    print(f"  {v}: {len(per_version[v])} files, same paths, blobs and sizes as the pinned version: {same}")
print("\nFiles at the pinned version:")
for p, (blob, size) in sorted(per_version[reach.DANDISET_VERSION].items()):
    print(f"  {p}  {size / 1e9:.2f} GB")
summary_meta = meta["assetsSummary"]
print("\nassetsSummary: subjects", summary_meta["numberOfSubjects"], "; species",
      [s["name"] for s in summary_meta["species"]], "; files", summary_meta["numberOfFiles"])
print("Description:", meta["description"])
print("Paper link:", [r["identifier"] for r in meta["relatedResource"] if r.get("relation") == "dcite:IsDescribedBy"])

The pinned version and the earlier published one hold the same ten files. The archive's summary
counts two subjects, as the files do, and lists the same species twice under two names.

In [ ]:
# Every file: subject, date, units, trials, conditions, and two faults found while loading
rows = []
for path in sorted(urls):
    f = h5py.File(remfile.File(urls[path]), "r")
    tr, u = f["intervals/trials"], f["units"]
    starts = tr["start_time"][:]
    index = u["spike_times_index"][:].astype(np.int64)
    fixed, wraps = reach.fix_first_index(u["spike_times"], index)
    cond = tr["maze_condition"][:] if "maze_condition" in tr else None
    rows.append({
        "file": path.split("/")[1].replace("_behavior+ecephys.nwb", ""),
        "session_start": f["session_start_time"][()].decode()[:10],
        "units": len(index), "trials": len(starts),
        "maze_conditions": int(len(np.unique(cond[cond > 0]))) if cond is not None else None,
        "has_correct_reach": "correct_reach" in tr,
        "spike_times_layout": "chunked gzip" if u["spike_times"].chunks else "contiguous",
        "index0_stored": int(index[0]), "index0_wraps_of_65536": wraps,
        "clock_resets": int(np.sum(np.diff(starts) < 0)),
        "trial_pairs_overlapping": int(np.sum(np.sort(starts)[1:] < tr["stop_time"][:][np.argsort(starts)][:-1])),
    })
files = pd.DataFrame(rows)
print(files.to_string(index=False))
print("\nUnits table description in every file:",
      {h5py.File(remfile.File(urls[p]), "r")["units"].attrs.get("description") for p in sorted(urls)[:1]})

Two faults of the files, found while loading and corrected in `reach.py`:

- In nine of the ten files (all but the 2010 file, which a later conversion wrote) the first entry
  of `units/spike_times_index` is the true end of unit 0 modulo 65536. Unit 0 is cut short and the
  rest of its spikes open unit 1, whose times then run from late in the session back to its start.
  The true end is the one position, at the stored value plus a multiple of 65536, where the times
  jump back from the last trial to the first; `fix_first_index` restores it (shown for J-array
  below). This is a format error (the index of a ragged column does not mark the end of its row).
- In five of the Nitschke files, including N-array (2010), the trial times go backwards at segment
  boundaries (`clock_resets`): the conversion code offsets only the first of the recording's
  segments, so later segments restart on the same clock and their trials overlap in time with
  earlier ones (`trial_pairs_overlapping`).
  Spike times per unit are therefore not sorted, and a spike cannot be placed in a trial by its
  time alone. The conversion wrote each unit's spikes trial by trial in table order, so
  `assign_to_trials` places them by that order; every spike of every unit lands in its trial's
  interval (it raises otherwise). Position samples are likewise one block per trial.

The units are one per electrode, 192 per file. The paper's J-array and N-array datasets had 146 and
218 isolations; the dandiset does not hold those, and its Units table has no description of what
its units are.

In [ ]:
for name, s in sessions.items():
    tr = s["trials"]
    sel = reach.select_trials(tr)
    cpc = tr[sel].maze_condition.value_counts()
    print(f"{name}: {len(tr)} trials; successful {int((tr.task_success == 1).sum())}; flagged discard "
          f"{int((tr.discard_trial == 1).sum())}; delay > 400 ms {int((tr.delay > 0.4).sum())}; "
          f"consistent {int(tr.correct_reach.sum())}; novel mazes (condition 0) {int((tr.maze_condition == 0).sum())}")
    print(f"   selected {int(sel.sum())} trials in {len(cpc)} conditions, {cpc.min()} to {cpc.max()} per condition (median {cpc.median():.0f}); "
          f"delay {tr.delay[sel].min():.3f} to {tr.delay[sel].max():.3f} s; "
          f"movement onset minus go cue {np.percentile((tr.move_begins_time - tr.go_cue_time)[sel], 5):.3f} to "
          f"{np.percentile((tr.move_begins_time - tr.go_cue_time)[sel], 95):.3f} s (5th to 95th percentile)")
    print(f"   condition 0 trials with novel_maze = 1: {int(((tr.maze_condition == 0) & (tr.novel_maze == 1)).sum())} of {int((tr.maze_condition == 0).sum())}")
    print(f"   session_start_time {s['meta']['session_start_time']}")

The Methods' selection (successful, consistent trials with delays above 400 ms) leaves 108
conditions in each session with at least 14 trials each. The delay is the go cue minus target onset;
the go-cue-to-movement interval (reaction time) lies within 0.29 to 0.49 s between the 5th and 95th
percentiles of every session, in the range of a reaching task. Condition 0 holds the novel mazes and is left out, as the paper
analyses 108 conditions.

In [ ]:
for name, s in sessions.items():
    tr = s["trials"]
    dur = (tr.stop_time - tr.start_time).sum()
    n_spk = np.array([len(x) for x in s["spikes"]])
    rate = n_spk / dur
    print(f"{name}: {len(s['spikes'])} units ({pd.Series(s['unit_area']).value_counts().to_dict()}), "
          f"electrode groups {pd.Series(s['unit_group']).value_counts().to_dict()}; electrode locations {s['meta']['electrode_locations']}")
    print(f"   one unit per electrode: {len(set(zip(s['unit_group'], range(len(s['spikes']))))) == len(s['spikes'])}; "
          f"rate over trial time: median {np.median(rate):.1f}, range {rate.min():.1f} to {rate.max():.1f} spikes/s")
    print(f"   obs_intervals equal the trial intervals: {np.allclose(s['obs_first'], tr[['start_time', 'stop_time']].values)}; "
          f"stored index wraps corrected: {s['index_wraps']}; spikes placed in their trials: {sum(o[-1] for o in s['spike_offsets'])} of {n_spk.sum()}")
s = sessions["J-array"]
st, fx = s["index_stored"], s["index_fixed"]
flat_counts = {"stored": (int(st[0]), int(st[1] - st[0])), "corrected": (int(fx[0]), int(fx[1] - fx[0]))}
print("\nJ-array spikes of unit 0 and unit 1, as stored and corrected:", flat_counts)
tr = s["trials"]
first_trial = lambda x: int(np.searchsorted(tr.start_time.values, x[0], "right") - 1)
last_trial = lambda x: int(np.searchsorted(tr.start_time.values, x[-1], "right") - 1)
print("Corrected: unit 0 spans trials", first_trial(s["spikes"][0]), "to", last_trial(s["spikes"][0]),
      "; unit 1 spans trials", first_trial(s["spikes"][1]), "to", last_trial(s["spikes"][1]),
      "; unit 1 times sorted:", bool(np.all(np.diff(s["spikes"][1]) >= 0)))

Each unit sits on its own electrode, 96 on the PMd array (group 1) and 96 on the M1 array (group 2),
and fires at 8.8 to 300 spikes/s over trial time (medians 33 to 36). These rates, and one unit
per electrode, are those of threshold crossings or of all spikes on a channel, not of sorted single
neurons; this is consistent with the data but the files do not say. Every spike of every unit is
placed in its own trial. After the correction, unit 0 spans the whole session and unit 1's times are
sorted.

In [ ]:
for name, s in sessions.items():
    tr = s["trials"]
    sel = np.where(reach.select_trials(tr))[0]
    xy = s["hand_xy"]
    print(f"{name}: hand position stated in '{s['meta']['hand_unit']}' (conversion {s['meta']['hand_conversion']}); "
          f"x {np.nanmin(xy[:, 0]):.1f} to {np.nanmax(xy[:, 0]):.1f}, y {np.nanmin(xy[:, 1]):.1f} to {np.nanmax(xy[:, 1]):.1f}; "
          f"targets x {tr.hit_x.min():.0f} to {tr.hit_x.max():.0f}, y {tr.hit_y.min():.0f} to {tr.hit_y.max():.0f}")
    rel, pos, vel, acc, jerk = reach.hand_kinematics(s, sel, tr.move_begins_time.values[sel])
    speed = np.median(np.linalg.norm(vel, axis=2), axis=0) * 1000
    print("   median hand speed (mm/s, values read as mm) at", {t: round(float(speed[t - reach.WINDOW_MS[0]]), 1) for t in (-200, -100, -50, 0, 50, 100, 200, 400)}, "ms from movement onset")
    off = s["hand_offsets"]
    ends = tr.move_ends_time.values[sel]
    ok = ~np.isnan(ends)
    idx = off[sel[ok]] + np.round((ends[ok] - s["hand_t"][off[sel[ok]]]) * 1000).astype(int)
    d_hand = xy[idx] - tr[["hit_x", "hit_y"]].values[sel[ok]]
    print(f"   hand at movement end minus hit target: median x {np.median(d_hand[:, 0]):.1f}, y {np.median(d_hand[:, 1]):.1f}")
    if s["cursor_xy"] is not None:
        d_cur = s["cursor_xy"][idx] - tr[["hit_x", "hit_y"]].values[sel[ok]]
        print(f"   cursor at movement end minus hit target: median x {np.median(d_cur[:, 0]):.1f}, y {np.median(d_cur[:, 1]):.1f}; "
              f"within the target half-width ({tr.target_size.iloc[0]}): {np.mean(np.all(np.abs(d_cur) <= tr.target_size.values[sel[ok], None], axis=1)):.2f}")

Hand position is stated in metres, but spans about -220 to +165 on the same scale as the target
positions (up to 145), and a reach peaks at about 600 to 730 per second: the values are in mm, the
units of the screen coordinates. The unit attribute is wrong. The hand ends a median 35 mm below
the hit target in J-array and 52 to 53 mm below it in the Nitschke sessions, while in J-array the
cursor lands on it (94% within the target), so hand and screen coordinates differ by an offset in y
that the files do not describe (the conversion code subtracts fixed offsets of 8 and 24 from hand y
for the two monkeys). Hand speed is near zero
100 ms before the trial table's movement onset and rises from it in all three sessions, so the hand data share the
trial table's clock.

In [ ]:
# Clock check with a contrast: decode the reach direction (octant of the hit target) from spike
# counts around movement onset, against the 200 ms before target onset and against the labels of the
# next trial. Chance is about 1/8.
rng = np.random.default_rng(7001)
rows = []
for name, s in sessions.items():
    tr = s["trials"]
    sel = np.where(reach.select_trials(tr))[0]
    y = (np.round(np.arctan2(tr.hit_y.values[sel], tr.hit_x.values[sel]) / (np.pi / 4)) % 8).astype(int)
    mv, tg = tr.move_begins_time.values[sel], tr.target_presentation_time.values[sel]
    X_move = reach.window_counts(s, sel, mv - 0.1, mv + 0.3)
    X_pre = reach.window_counts(s, sel, tg - 0.2, tg)
    seg = np.r_[0, np.cumsum(np.diff(tr.start_time.values) < 0)][sel]
    row = {"session": name, "trials": len(sel), "largest class": np.bincount(y).max() / len(y),
           "movement window": reach.decode_accuracy(X_move, y, rng),
           "before target": reach.decode_accuracy(X_pre, y, rng),
           "next trial's labels": reach.decode_accuracy(X_move[:-1], y[1:], rng)}
    for g in np.unique(seg):
        m = seg == g
        row[f"clock segment {g} ({m.sum()} trials)"] = reach.decode_accuracy(X_move[m], y[m], rng)
    rows.append(row)
print(pd.DataFrame(rows).set_index("session").T.round(3).to_string())

In every session and in every clock segment of the 2010 and 20090910 files, the reach direction is
decoded from the spikes around the trial table's movement onset far above chance, while the 200 ms
before target onset and the next trial's labels give about chance. Spikes and trial events share a
clock in every segment, and the placement of spikes by storage order puts them in the right trials.

In [ ]:
# Which session is the "N-9-10-2010" dataset of Supplementary Figure 5? Compare the reached target
# of each maze condition (the most common hit target over its successful trials).
def condition_targets(s):
    tr = s["trials"]
    ok = (tr.task_success == 1) & (tr.maze_condition > 0)
    return tr[ok].groupby("maze_condition")[["hit_x", "hit_y"]].agg(lambda v: v.mode().iloc[0])
targets = {name: condition_targets(s) for name, s in sessions.items()}
for a, b in [("N-0910", "N-array"), ("N-0910", "J-array"), ("N-array", "J-array")]:
    common = targets[a].index.intersection(targets[b].index)
    same = (targets[a].loc[common] == targets[b].loc[common]).all(axis=1)
    print(f"{a} against {b}: {len(common)} conditions in both; same reached target {same.mean():.2f}")
print("Session dates in the files:", {n: s["meta"]["session_start_time"][:10] for n, s in sessions.items()})

`sub-Nitschke_ses-20090910` has 108 conditions, and for all of them the same reached target
as N-array (2010), while neither shares any with J-array. The other Nitschke sessions of 2009 have
27 conditions or none. The conversion code
that wrote it (`maze_task/conversion_script.py`) builds every session date as "2009" plus the folder
name, so a session recorded on 10 September 2010 would be dated 2009-09-10. This is consistent with
the file being the "N-9-10-2010" dataset of Supplementary Figure 5, which is otherwise absent, but it
is not shown; it is used here only as a variant.

## Firing rates and jPCA

Shared by the claims below. Rates are averaged per condition over the selected trials, aligned on
movement onset, and smoothed with a 20 ms Gaussian. jPCA uses -50 to +150 ms every 10 ms and 6 PCs.

In [ ]:
prep = {}
for name, s in sessions.items():
    tr = s["trials"]
    sel = np.where(reach.select_trials(tr))[0]
    align = tr.move_begins_time.values[sel]
    cond = tr.maze_condition.values[sel]
    labels, sums, n = reach.condition_counts(s, sel, align, cond)
    prep[name] = {"sel": sel, "align": align, "cond": cond, "labels": labels, "sums": sums, "n": n,
                  "rates": reach.condition_rates(sums, n)}
res = {name: reach.jpca(p["rates"]) for name, p in prep.items()}
main = pd.DataFrame({name: reach.summary(r) for name, r in res.items()}).T
print(main.round(3).to_string())
print("Rates (spikes/s) over all conditions and times: ",
      {n: (round(float(p["rates"].min()), 1), round(float(p["rates"].max()), 1)) for n, p in prep.items()})

In [ ]:
# Variants used in several claims: 24 ms smoothing, other windows, all successful trials (not only
# consistent ones), and 10 PCs.
variants = {}
for name in ("J-array", "N-array"):
    p, s = prep[name], sessions[name]
    v = {"main": reach.summary(res[name]),
         "24 ms Gaussian": reach.summary(reach.jpca(reach.condition_rates(p["sums"], p["n"], 24.0))),
         "window -100 to +100 ms": reach.summary(reach.jpca(p["rates"], times_ms=np.arange(-100, 101, 10))),
         "window 0 to +200 ms": reach.summary(reach.jpca(p["rates"], times_ms=np.arange(0, 201, 10))),
         "10 PCs": reach.summary(reach.jpca(p["rates"], n_pcs=10))}
    tr = s["trials"]
    sel_all = np.where(reach.select_trials(tr, consistent=False))[0]
    lab, sums_all, n_all = reach.condition_counts(s, sel_all, tr.move_begins_time.values[sel_all], tr.maze_condition.values[sel_all])
    v[f"all successful trials ({len(sel_all)})"] = reach.summary(reach.jpca(reach.condition_rates(sums_all, n_all)))
    variants[name] = pd.DataFrame(v).T
    print(name)
    print(variants[name].round(3).to_string(), "\n")

In [ ]:
# Shuffle controls (Supplementary Figures 2 and 3), used by suppfig3-shuffle-controls and
# fig3-phase-from-prep-state. Dividing point 50 ms after the go cue: the median go-cue time
# relative to movement onset plus 50 ms.
rng_shuffle = np.random.default_rng(7012)
shuffles = {}
for name in ("J-array", "N-array"):
    p, tr = prep[name], sessions[name]["trials"]
    go_rel = np.median((tr.go_cue_time - tr.move_begins_time).values[p["sel"]]) * 1000
    t0 = int(round(go_rel + 50))
    t0_index = t0 - reach.WINDOW_MS[0]
    out = []
    for kind, reps in ((1, 20), (2, 1), (3, 20)):
        for r in range(reps):
            sh = reach.jpca(reach.shuffle_rates(p["rates"], t0_index, kind, rng_shuffle))
            out.append({"shuffle": kind, "run": r, **reach.summary(sh), "prep_r2": reach.prep_prediction_r2(sh["planes"][0])})
    shuffles[name] = {"t0_ms": t0, "runs": pd.DataFrame(out)}
    print(f"{name}: dividing point {t0} ms from movement onset")

## array-dataset-composition

> The two array datasets, monkey J-array recorded 2009-09-18 and monkey N-array recorded
> 2010-09-23, each present 108 reach conditions and comprise 146 and 218 isolations.

In [ ]:
comp = pd.DataFrame({name: {"session_start": sessions[name]["meta"]["session_start_time"][:10],
                            "conditions": len(prep[name]["labels"]),
                            "units in the dandiset": len(sessions[name]["spikes"]),
                            "isolations in the paper": paper_n}
                     for name, paper_n in (("J-array", 146), ("N-array", 218))}).T
print(comp.to_string())
print("Tolerance on isolations (10%): J-array 131.4 to 160.6, N-array 196.2 to 239.8")

**Result: partially reproduced.** Both sessions are in the dandiset with 108 conditions each. Each holds 192 units, one per electrode, against the paper's 146 (J-array) and 218 (N-array) isolations, outside the 10% tolerance in both. The sorted isolations the paper analysed are not in the dandiset, so every claim below is tested on the 192 units.

## fig3ef-same-direction

> In the first jPCA plane the neural state rotates in the same direction for all conditions of the
> J-array and N-array datasets, including reaches in opposite directions.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
for ax, name in zip(axes, ("J-array", "N-array")):
    proj = res[name]["planes"][0]
    col = plt.cm.RdYlGn_r((proj[:, 0, 0] - proj[:, 0, 0].min()) / np.ptp(proj[:, 0, 0]))
    for c in range(proj.shape[0]):
        ax.plot(proj[c, :, 0], proj[c, :, 1], color=col[c], lw=0.8)
        ax.plot(proj[c, 0, 0], proj[c, 0, 1], "o", color=col[c], ms=3)
    ax.set_title(f"{name}, -50 to +150 ms")
    ax.set_xlabel("jPC1 (a.u.)"); ax.set_ylabel("jPC2 (a.u.)")
    print(f"{name}: plotted jPC1 {proj[..., 0].min():.3f} to {proj[..., 0].max():.3f}, jPC2 {proj[..., 1].min():.3f} to {proj[..., 1].max():.3f} (all points inside the axes)")
plt.tight_layout(); plt.show()
for name in ("J-array", "N-array"):
    rot = reach.net_rotation(res[name]["planes"][0])
    print(f"{name}: {np.sum(rot > 0)} of {len(rot)} conditions rotate counter-clockwise ({np.mean(rot > 0):.3f}); "
          f"net rotation median {np.degrees(np.median(rot)):.1f} deg, 5th to 95th percentile "
          f"{np.degrees(np.percentile(rot, 5)):.1f} to {np.degrees(np.percentile(rot, 95)):.1f} deg")
print("\nFraction counter-clockwise under the variants:")
print(pd.DataFrame({n: variants[n].frac_ccw for n in variants}).round(3).to_string())

**Result: reproduced.** In the first jPCA plane 102 of 108 conditions (94.4%) rotate counter-clockwise in J-array and 108 of 108 in N-array, with median net rotations of 70.2 and 114.6 degrees over the 200 ms. Under the variants the fraction stays between 0.944 and 0.981 (J-array) and between 0.972 and 1.000 (N-array). jPCA orients the plane so that the net rotation is counter-clockwise, so this fraction is also high where no rotation was put in: 0.963 to 0.991 after shuffles 2 and 3, and 0.926 and 0.972 for the velocity model with no lead. It shows that the conditions turn the same way, not on its own that the rotation is specific to the data.

## fig3-independent-of-curvature

> The rotations do not relate directly to reach curvature; the neural state rotates the same way for
> clockwise-curving, counter-clockwise-curving and straight reaches.

In [ ]:
kin = {name: reach.condition_kinematics(sessions[name], prep[name]["sel"], prep[name]["align"], prep[name]["cond"])
       for name in ("J-array", "N-array")}

def logistic_slope(x, y):
    if y.all() or not y.any():
        return None
    xs = (x - x.mean()) / x.std()
    nll = lambda b: np.sum(np.logaddexp(0, b[0] + b[1] * xs) - y * (b[0] + b[1] * xs))
    return minimize(nll, np.zeros(2)).x[1] / x.std()

for name in ("J-array", "N-array"):
    turn = reach.heading_change(kin[name]["vel"])
    rot = reach.net_rotation(res[name]["planes"][0])
    cls = np.where(turn < -20, "clockwise", np.where(turn > 20, "counter-clockwise", "straight"))
    tab = pd.DataFrame({"class": cls, "ccw": rot > 0}).groupby("class").ccw.agg(["size", "mean"])
    print(f"{name}: heading change of the mean hand path {turn.min():.0f} to {turn.max():.0f} deg")
    print(tab.rename(columns={"size": "conditions", "mean": "fraction rotating ccw"}).round(3).to_string())
    rho, p = stats.spearmanr(turn, rot)
    slope = logistic_slope(turn, (rot > 0).astype(float))
    print(f"   Spearman correlation of heading change with net neural rotation: rho {rho:.3f}, p {p:.3g}; "
          f"logistic slope of rotation direction on heading change: {'not estimable (all conditions rotate one way)' if slope is None else f'{slope:.4f} per deg'}\n")

In [ ]:
# The hand paths by class (dotted: conditions whose neural state rotates clockwise), and the
# classes when the heading is taken over the unbroken run of samples around peak speed only, or
# with thresholds of 10 and 30 degrees.
fig, axes = plt.subplots(1, 2, figsize=(12, 6))
colours = {"clockwise": "tab:red", "counter-clockwise": "tab:blue", "straight": "0.3"}
rows = []
for ax, name in zip(axes, ("J-array", "N-array")):
    pos, vel = kin[name]["pos"], kin[name]["vel"]
    rot = reach.net_rotation(res[name]["planes"][0])
    turn = reach.heading_change(vel)
    cls = np.where(turn < -20, "clockwise", np.where(turn > 20, "counter-clockwise", "straight"))
    for c in range(len(turn)):
        xy = pos[c, -reach.WINDOW_MS[0]:] * 1000
        ax.plot(xy[:, 0], xy[:, 1], color=colours[cls[c]], lw=1.2 if rot[c] < 0 else 0.6, ls="-" if rot[c] > 0 else ":")
    ax.set_title(f"{name}: mean hand path, movement onset to +500 ms"); ax.set_xlabel("x (mm)"); ax.set_ylabel("y (mm)"); ax.set_aspect("equal")
    print(f"{name}: plotted x {pos[:, -reach.WINDOW_MS[0]:, 0].min() * 1000:.0f} to {pos[:, -reach.WINDOW_MS[0]:, 0].max() * 1000:.0f}, "
          f"y {pos[:, -reach.WINDOW_MS[0]:, 1].min() * 1000:.0f} to {pos[:, -reach.WINDOW_MS[0]:, 1].max() * 1000:.0f} mm")
    print("   conditions rotating clockwise (heading change, net neural rotation, deg):",
          [(int(kin[name]["labels"][c]), round(float(turn[c])), round(float(np.degrees(rot[c])))) for c in np.where(rot < 0)[0]])
    for method, tc in (("all samples above 10% of peak speed", turn), ("unbroken run around peak speed", reach.heading_change(vel, contiguous=True))):
        for thr in (10, 20, 30):
            k = np.where(tc < -thr, "clockwise", np.where(tc > thr, "counter-clockwise", "straight"))
            for c in ("clockwise", "counter-clockwise", "straight"):
                rows.append({"session": name, "heading": method, "threshold deg": thr, "class": c,
                             "conditions": int((k == c).sum()), "fraction rotating ccw": float(np.mean(rot[k == c] > 0)) if (k == c).any() else np.nan})
plt.tight_layout(); plt.show()
print(pd.DataFrame(rows).pivot_table(index=["session", "heading", "threshold deg"], columns="class",
                                     values=["conditions", "fraction rotating ccw"]).round(3).to_string())

**Result: partially reproduced.** In N-array every condition rotates counter-clockwise in all three classes of reach (44 clockwise, 53 counter-clockwise, 11 straight). In J-array 98.4% of the 63 counter-clockwise and all 7 straight reaches do, but 86.8% of the 38 clockwise reaches, below the 90% the criterion asks and above 75%. Rotation and curvature are weakly related, with opposite signs in the two datasets (Spearman rho 0.227 in J-array, -0.214 in N-array); in J-array the logistic slope of rotation direction on heading change is 0.0101 per degree. The hand paths drawn by class match the classes: the five clockwise-class conditions of J-array whose neural state turns clockwise are loops of 162 to 180 degrees, with net neural rotations of -1 to -21 degrees in four of them and -89 degrees in the fifth. With the heading taken over the unbroken run around peak speed, or with thresholds of 10 or 30 degrees, the J-array clockwise class has 38 or 41 conditions and 0.868 to 0.878 of them rotate counter-clockwise, so the outcome does not depend on how the classes are drawn.

## fig3-phase-from-prep-state

> The rotation phase follows from the preparatory state; knowing the initial state in the jPC plane,
> the subsequent states are reasonably predictable.

In [ ]:
for name in ("J-array", "N-array"):
    r2 = reach.prep_prediction_r2(res[name]["planes"][0])
    sh3 = shuffles[name]["runs"].query("shuffle == 3").prep_r2
    sh = {k: shuffles[name]["runs"].query("shuffle == @k").prep_r2.median() for k in (1, 2, 3)}
    print(f"{name}: leave-one-condition-out R2 of the +150 ms state from the -50 ms state {r2:.3f}; "
          f"shuffle 3 median {sh3.median():.3f} (range {sh3.min():.3f} to {sh3.max():.3f}); shuffle medians {({k: round(float(v), 3) for k, v in sh.items()})}")
for name in ("J-array", "N-array"):
    print(f"{name} variants:", {k: round(reach.prep_prediction_r2(reach.jpca(prep[name]['rates'], times_ms=t)['planes'][0]), 3)
                                for k, t in (("-100 to +100", np.arange(-100, 101, 10)), ("0 to +200", np.arange(0, 201, 10)))})

In [ ]:
# The R2 above per shuffle run (shuffle 1 against shuffle 3), and the same prediction with jPCA
# refitted leaving each condition out, together with a score of the phase alone: the last phase
# predicted as the first plus the other conditions' mean phase advance (`prep_prediction_cv`).
for name in ("J-array", "N-array"):
    runs = shuffles[name]["runs"]
    print(f"{name}, in-sample R2 per run (data {reach.prep_prediction_r2(res[name]['planes'][0]):.3f}):")
    print(pd.DataFrame({f"shuffle {k}": runs.query("shuffle == @k").prep_r2.values for k in (1, 3)}).T.round(3).to_string(), "\n")
rng_cv = np.random.default_rng(7018)
rows = []
for name in ("J-array", "N-array"):
    p = prep[name]
    t0_index = shuffles[name]["t0_ms"] - reach.WINDOW_MS[0]
    rows.append({"session": name, "rates": "data", "run": 0, **reach.prep_prediction_cv(p["rates"])})
    for kind, reps in ((1, 10), (2, 1), (3, 10)):
        for r in range(reps):
            rows.append({"session": name, "rates": f"shuffle {kind}", "run": r,
                         **reach.prep_prediction_cv(reach.shuffle_rates(p["rates"], t0_index, kind, rng_cv))})
cv = pd.DataFrame(rows)
print(cv.groupby(["session", "rates"]).agg(runs=("r2", "size"), r2_median=("r2", "median"), r2_min=("r2", "min"), r2_max=("r2", "max"),
                                           phase_median=("phase_score", "median"), phase_min=("phase_score", "min"), phase_max=("phase_score", "max"),
                                           static_median=("static_phase_score", "median"), advance_deg_median=("advance_deg", "median")).round(3).to_string())
for name in ("J-array", "N-array"):
    d, s3 = cv.query("session == @name and rates == 'data'").iloc[0], cv.query("session == @name and rates == 'shuffle 3'")
    print(f"{name}: shuffle 3 runs at or above the data, cross-validated R2 {int((s3.r2 >= d.r2).sum())} of {len(s3)}, phase score {int((s3.phase_score >= d.phase_score).sum())} of {len(s3)}")

**Result: partially reproduced.** One complex gain predicts the state at +150 ms from the state at -50 ms with cross-validated R2 0.632 (J-array) and 0.748 (N-array), both above 0.5. In N-array this is above every shuffle 3 run (median 0.656); in J-array it is below every shuffle 3 run (0.633 to 0.718, median 0.671). Shuffle 1, which has no rotation, scores higher than the data in every run (0.82 to 0.91), so this R2 rewards any consistent map from the first state to the last and is a weak test of the claim. With jPCA refitted leaving each condition out, the R2 is 0.615 and 0.727, and the pattern is the same: 8 of 10 shuffle 3 runs reach the J-array value and none the N-array value. The phase score, which asks whether the last phase follows from the first plus a shared advance, separates rotation from no rotation: the data score 0.763 (J-array) and 0.894 (N-array) against 0.236 and -0.356 for a prediction of no advance, while for shuffle 1 the two scores are equal (median 0.78 and 0.73). Shuffle 3 keeps a shared advance (median 65 and 99 degrees) and scores 0.811 and 0.797; the data exceed it in N-array and not in J-array, as for the criterion's measure.


## var-captured-jpc-plane

> The first jPCA plane captures an average of 28% of the total data variance (typically 20 to 35%).

In [ ]:
pv = main.loc[["J-array", "N-array"], "plane_var"] * 100
print("Variance captured by the first jPCA plane (%):", pv.round(1).to_dict(), "; mean of the two", round(pv.mean(), 1))
print("Reading (a), mean within 25.2 to 30.8:", 25.2 <= pv.mean() <= 30.8, "; reading (b), each within 18 to 38.5:", bool(((pv >= 18) & (pv <= 38.5)).all()))
print("N-0910 (variant):", round(main.loc["N-0910", "plane_var"] * 100, 1))
print("\nUnder the variants (%):")
print((pd.DataFrame({n: variants[n].plane_var for n in variants}) * 100).round(1).to_string())

**Result: reproduced.** The first jPCA plane captures 20.0% (J-array) and 23.1% (N-array) of the variance, both within the stated typical range widened to 18 to 38.5% (reading b). Their mean, 21.6%, is below the paper's average of 28 +- 2.8% (reading a). With 6 PCs the variants give 17.5 to 20.9% (J-array) and 20.8 to 24.8% (N-array). These are the dandiset's per-electrode units, not the paper's sorted isolations, so the fractions of variance compare different populations.

## pc23-variance

> The plane of PC2 and PC3 captures 29% of the total variance on average, about as much as the first
> jPCA plane.

In [ ]:
t = main.loc[["J-array", "N-array"], ["pc1_var", "pc23_var", "plane_var", "pcs_var"]] * 100
t["jPC plane / PC2+PC3"] = t.plane_var / t.pc23_var
print(t.round(2).to_string())
print("Mean PC2+PC3 of the two datasets:", round(t.pc23_var.mean(), 1), "% (expected 29 +- 2.9)")
print("\nUnder the variants, PC2+PC3 (%):")
print((pd.DataFrame({n: variants[n].pc23_var for n in variants}) * 100).round(1).to_string())

**Result: partially reproduced.** PC2 and PC3 together capture 21.6% (J-array) and 27.0% (N-array), mean 24.3%, below 29 +- 2.9% under both readings (N-array alone is within it). The jPCA plane captures 0.93 (J-array) and 0.86 (N-array) times the variance of PC2 and PC3, so the second part of the claim holds. These are the dandiset's per-electrode units, not the paper's sorted isolations, so the fractions of variance compare different populations.

## suppfig4-j-array-planes and suppfig4-planes-range

> In J-array the first three jPCA planes found within the top 10 PCs capture 22%, 19% and 12% of the
> variance (54% together), against 64% for the top six PCs.
>
> Every dataset contains two, and usually three, planes with rotations, together capturing 50 to 70%
> of the variance.

In [ ]:
planes10 = {}
for name in ("J-array", "N-array", "N-0910"):
    r10 = reach.jpca(prep[name]["rates"], n_pcs=10, n_planes=3)
    planes10[name] = {"plane 1 %": r10["plane_var"][0] * 100, "plane 2 %": r10["plane_var"][1] * 100,
                      "plane 3 %": r10["plane_var"][2] * 100, "three planes %": r10["plane_var"].sum() * 100,
                      "top 6 PCs %": res[name]["pc_var"].sum() * 100, "top 10 PCs %": r10["pc_var"].sum() * 100,
                      **{f"plane {k + 1} ccw": float(np.mean(reach.net_rotation(r10["planes"][k]) > 0)) for k in range(3)},
                      **{f"plane {k + 1} Hz": r10["eigenvalues_rad_s"][k] / (2 * np.pi) for k in range(3)}}
print(pd.DataFrame(planes10).T.round(3).to_string())
print("\nExpected for J-array: 22 +- 2.2, 19 +- 1.9, 12 +- 1.2, total 54 +- 5.4, top six PCs 64 +- 6.4; range for every dataset 45 to 77% (N-0910 is a variant)")

**Result: not reproduced.** In J-array the first three jPCA planes from the top 10 PCs capture 12.3%, 14.8% and 12.8% (40.0% together), against 22%, 19% and 12% (54%) in the paper; only the third plane is within tolerance. The top six PCs capture 60.5%, within 64 +- 6.4%. All three planes rotate consistently, with 92.6% to 99.1% of conditions counter-clockwise. These are the dandiset's per-electrode units, not the paper's sorted isolations, so the fractions of variance compare different populations.

**Result: partially reproduced.** The three planes capture 50.1% of the variance in N-array, inside the widened range of 45 to 77%, and 40.0% in J-array, below it. In both datasets the first two planes have 95.4% or more of the conditions rotating in the plane's direction. In ses-20090910 the three planes capture 40.9%. These are the dandiset's per-electrode units, not the paper's sorted isolations, so the fractions of variance compare different populations.

## fig6a-angle-peak

> The distribution of the angle between the neural state and its derivative in the first jPCA plane
> peaks near pi/2.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
for name in ("J-array", "N-array"):
    ang = reach.state_angles(res[name]["planes"][0])
    h, e = np.histogram(ang, bins=36, range=(-np.pi, np.pi))
    ax.plot((e[:-1] + e[1:]) / 2 / np.pi, h, label=name)
    print(f"{name}: {ang.size} angles; peak {reach.angle_peak(ang) / np.pi:.3f} pi; median {np.median(ang) / np.pi:.3f} pi; "
          f"fraction within pi/4 of pi/2 {np.mean(np.abs(ang - np.pi / 2) < np.pi / 4):.3f}; peak with 24 and 48 bins "
          f"{reach.angle_peak(ang, 24) / np.pi:.3f} and {reach.angle_peak(ang, 48) / np.pi:.3f} pi; largest bin count {h.max()}")
ax.set_xlabel("angle between x and dx/dt (units of pi)"); ax.set_ylabel("count"); ax.legend(); plt.show()
print("\nPeak (units of pi) under the variants:")
print((pd.DataFrame({n: variants[n].angle_peak for n in variants}) / np.pi).round(3).to_string())

**Result: reproduced.** The angle between the state and its derivative peaks at 0.528 pi in J-array and 0.639 pi in N-array (medians 0.511 and 0.603 pi), within 0.3 to 0.7 pi. Under the variants the peak stays between 0.528 and 0.583 pi (J-array) and between 0.583 and 0.639 pi (N-array), and with 24 or 48 bins it lies between 0.542 and 0.646 pi.

## fig6b-skew-fit

> For the neural data the rotational dynamics matrix fits the data nearly as well as the best
> unconstrained linear dynamics matrix, in 6-D and in the first jPCA plane.

In [ ]:
print(main.loc[["J-array", "N-array", "N-0910"], ["r2_skew", "r2_best", "r2_ratio_6d", "r2_ratio_plane"]].round(3).to_string())
print("\n6-D ratio under the variants:")
print(pd.DataFrame({n: variants[n].r2_ratio_6d for n in variants}).round(3).to_string())
print("\nPlane ratio under the variants:")
print(pd.DataFrame({n: variants[n].r2_ratio_plane for n in variants}).round(3).to_string())

**Result: reproduced.** R2(M_skew) / R2(M) is 0.905 (J-array) and 0.845 (N-array) in 6-D and 0.957 and 0.881 in the first plane, above the thresholds of 0.7 and 0.75. Under the variants the ratios stay at 0.819 or more in 6-D and 0.879 or more in the plane. The absolute R2 of M_skew is 0.343 (J-array) and 0.517 (N-array).

## rotation-frequency

> The rotations are brief and fast, up to about 2.5 Hz; the largest eigenvalues of M_skew in the
> monkey A and B datasets are 8.8 to 13.5 radians/s.

In [ ]:
for name in ("J-array", "N-array", "N-0910"):
    ev = res[name]["eigenvalues_rad_s"]
    print(f"{name}: eigenvalue magnitudes of M_skew (rad/s) {np.round(ev, 2).tolist()}; first plane {ev[0] / (2 * np.pi):.2f} Hz")
print("\nFirst-plane frequency (Hz) under the variants:")
print(pd.DataFrame({n: variants[n].freq_hz for n in variants}).round(2).to_string())

In [ ]:
# Is J-array's low frequency a slower rotation, or a rotation that starts after the window does?
# The first plane of the main fit, projected from -300 to +300 ms; its mean angular speed over the
# conditions at each time; and the fit with 200 ms windows starting at other times.
times_long = np.arange(-300, 301, 10)
fig, axes = plt.subplots(3, 2, figsize=(11, 8), sharex=True)
for col, name in enumerate(("J-array", "N-array")):
    r = res[name]
    proj = reach.project(prep[name]["rates"], {**r, "transform": {**r["transform"], "times_ms": times_long}})
    x, dx = proj[:, :-1], np.diff(proj, axis=1)
    ang_speed = np.mean((x[..., 0] * dx[..., 1] - x[..., 1] * dx[..., 0]) / np.sum(x ** 2, axis=2), axis=0) / reach.DT_S / (2 * np.pi)
    for k in range(2):
        axes[k, col].plot(times_long, proj[:, :, k].T, lw=0.4, color="0.3", alpha=0.5)
        axes[k, col].axvspan(-50, 150, color="tab:orange", alpha=0.15)
        axes[k, col].set_ylabel(f"jPC{k + 1} (a.u.)")
    axes[0, col].set_title(f"{name}, first plane of the -50 to +150 ms fit")
    axes[2, col].plot(times_long[:-1] + 5, ang_speed); axes[2, col].axhline(0, color="0.6", lw=0.5)
    axes[2, col].axvspan(-50, 150, color="tab:orange", alpha=0.15)
    axes[2, col].set_ylabel("mean angular speed (Hz)"); axes[2, col].set_xlabel("ms from movement onset")
    print(f"{name}: jPC1 {proj[..., 0].min():.3f} to {proj[..., 0].max():.3f}, jPC2 {proj[..., 1].min():.3f} to {proj[..., 1].max():.3f}; "
          "mean angular speed (Hz) at", {int(t + 5): round(float(v), 2) for t, v in zip(times_long[:-1], ang_speed) if t % 50 == 0})
plt.tight_layout(); plt.show()
rows = []
for name in ("J-array", "N-array"):
    for start in (-150, -100, -50, 0, 50):
        sm = reach.summary(reach.jpca(prep[name]["rates"], times_ms=np.arange(start, start + 201, 10)))
        rows.append({"session": name, "window": f"{start} to {start + 200} ms", "freq_hz": sm["freq_hz"],
                     "plane_var": sm["plane_var"], "frac_ccw": sm["frac_ccw"], "r2_ratio_6d": sm["r2_ratio_6d"]})
print(pd.DataFrame(rows).round(3).to_string(index=False))

# Helper check: a steady 2 Hz rotation in 192 units, present throughout or starting at movement onset
rng_syn = np.random.default_rng(7019)
t_all = np.arange(*reach.WINDOW_MS)
phase0, mix = rng_syn.uniform(0, 2 * np.pi, 108), rng_syn.normal(size=(2, 192))
for label, start in (("throughout", reach.WINDOW_MS[0]), ("from movement onset", 0)):
    ang = phase0[:, None] + 2 * np.pi * 2.0 * np.clip(t_all - start, 0, None) / 1000
    syn_rates = 20 + 10 * np.stack([np.cos(ang), np.sin(ang)], axis=-1) @ mix + rng_syn.normal(0, 0.5, (108, len(t_all), 192))
    f = {w: round(reach.jpca(syn_rates, times_ms=np.arange(w, w + 201, 10))["freq_hz"], 2) for w in (-50, 0)}
    print(f"Synthetic 2 Hz rotation {label}: frequency with the window from -50 ms {f[-50]} Hz, from 0 ms {f[0]} Hz")
del syn_rates

**Result: partially reproduced.** The first plane rotates at 1.38 Hz in N-array (8.65 radians/s), inside 1.26 to 2.75 Hz, and at 1.03 Hz in J-array (6.49 radians/s), below it. No variant exceeds 1.68 Hz, so the upper bound of about 2.5 Hz holds; J-array reaches the range only with the -100 to +100 ms window (1.45 Hz) or 10 PCs (1.68 Hz). J-array's lower frequency is not the effect of a window that starts before a rotation does (on a synthetic 2 Hz rotation that starts at movement onset, the -50 to +150 ms window reads 1.49 Hz and a window from 0 ms reads 1.99 Hz): in J-array windows starting at 0 or +50 ms give 1.04 and 1.13 Hz. Its mean angular speed in the plane is about 1.3 Hz from -60 to +20 ms and falls to 0.7 to 0.9 Hz by +100 ms, and windows starting at -150 or -100 ms give 1.31 and 1.45 Hz, so J-array rotates faster before movement onset than over the window used here. The paper chose a window per dataset; a J-array window starting before -50 ms would reach the range.

## suppfig3-shuffle-controls

> Three shuffle controls that keep preparatory activity and invert or reassign the peri-movement
> activity largely remove the rotational structure found by jPCA.

In [ ]:
for name in ("J-array", "N-array"):
    runs = shuffles[name]["runs"]
    orig = main.loc[name]
    g = runs.groupby("shuffle").agg(runs=("r2_skew", "size"), r2_skew_median=("r2_skew", "median"), r2_skew_max=("r2_skew", "max"),
                                    ratio_6d_median=("r2_ratio_6d", "median"), frac_ccw_median=("frac_ccw", "median"),
                                    plane_var_median=("plane_var", "median"))
    g["all runs below original"] = runs.groupby("shuffle").r2_skew.max() < orig.r2_skew
    g["median / original"] = g.r2_skew_median / orig.r2_skew
    print(f"{name}: original R2(M_skew) {orig.r2_skew:.3f}, 6-D ratio {orig.r2_ratio_6d:.3f}, fraction ccw {orig.frac_ccw:.3f}; dividing point {shuffles[name]['t0_ms']} ms")
    print(g.round(3).to_string(), "\n")
fig, axes = plt.subplots(1, 4, figsize=(14, 3.5))
p = prep["J-array"]
t0_index = shuffles["J-array"]["t0_ms"] - reach.WINDOW_MS[0]
demo_rng = np.random.default_rng(7013)
for ax, kind in zip(axes, (0, 1, 2, 3)):
    rates = p["rates"] if kind == 0 else reach.shuffle_rates(p["rates"], t0_index, kind, demo_rng)
    proj = reach.jpca(rates)["planes"][0]
    for c in range(proj.shape[0]):
        ax.plot(proj[c, :, 0], proj[c, :, 1], lw=0.6)
    ax.set_title("J-array original" if kind == 0 else f"J-array shuffle {kind}")
    print(f"panel {kind}: jPC1 {proj[..., 0].min():.3f} to {proj[..., 0].max():.3f}, jPC2 {proj[..., 1].min():.3f} to {proj[..., 1].max():.3f}")
plt.tight_layout(); plt.show()

**Result: not reproduced.** Shuffle 1 removes the rotational structure: R2(M_skew) falls to a median of 1.0% (J-array) and 0.8% (N-array) of the original, and the fraction of conditions rotating counter-clockwise to 0.565 and 0.602. Shuffles 2 and 3 do not meet the criterion. In J-array they leave 71% and 81% of the original R2(M_skew) (shuffle 2 is under the 75% threshold, shuffle 3 is not); in N-array they leave 94% each, and one run of shuffle 3 exceeds the original. After shuffles 2 and 3 the median fraction rotating counter-clockwise is 0.963 to 0.991 in both datasets. The dividing point here is one time per dataset relative to movement onset (-301 and -320 ms), where the paper used 50 ms after the go cue of each condition's own alignment. As the supplement says, shuffles 2 and 3 are not expected to remove all rotational structure; they are meant to break the relation of the rotation to the preparatory state, which R2(M_skew) does not measure. The criterion fixed R2(M_skew) before the analysis, and the outcome follows it.

## suppfig5-areas-variance and suppfig5-m1-faster

> Analysed separately, the PMd and M1 arrays each show clear rotations; the jPCA plane captures 29%
> (PMd) and 29% (M1) in J-array and 32% and 28% in N-array.
>
> The angular velocity of the rotations is slightly higher for M1 than for PMd in each dataset.

In [ ]:
rows = []
for name in ("J-array", "N-array", "N-0910"):
    for area in ("PMd", "M1"):
        m = sessions[name]["unit_area"] == area
        for label, rates in (("main", prep[name]["rates"]), ("24 ms Gaussian", reach.condition_rates(prep[name]["sums"], prep[name]["n"], 24.0))):
            for wlabel, times in (("-50 to +150", reach.ANALYSIS_TIMES), ("-100 to +100", np.arange(-100, 101, 10))):
                r = reach.jpca(rates[:, :, m], times_ms=times)
                sm = reach.summary(r)
                rows.append({"session": name, "area": area, "units": int(m.sum()), "smoothing": label, "window": wlabel,
                             "plane var %": sm["plane_var"] * 100, "frac ccw": sm["frac_ccw"], "freq Hz": sm["freq_hz"],
                             "r2 ratio 6d": sm["r2_ratio_6d"]})
areas = pd.DataFrame(rows)
print(areas.round(3).to_string(index=False))
mainrows = areas[(areas.smoothing == "main") & (areas.window == "-50 to +150")].set_index(["session", "area"])
print("\nExpected (%): J-array PMd 29, M1 29; N-array PMd 32, M1 28; N-9-10-2010 PMd 26, M1 24 (variant)")
for name in ("J-array", "N-array", "N-0910"):
    f = mainrows.loc[name, "freq Hz"]
    print(f"{name}: M1 {f['M1']:.2f} Hz, PMd {f['PMd']:.2f} Hz, M1 higher: {f['M1'] > f['PMd']}")
v = areas.pivot_table(index=["session", "smoothing", "window"], columns="area", values="freq Hz")
v["M1 higher"] = v.M1 > v.PMd
print("\nM1 against PMd frequency under the variants:")
print(v.round(2).to_string())

**Result: partially reproduced.** The jPCA plane captures 23.7% (PMd) and 22.8% (M1) in J-array and 18.1% and 25.0% in N-array, against 29, 29, 32 and 28%; none is within the 10% tolerance (N-array M1 misses its lower bound of 25.2% by 0.2 points). All four lie within the widened typical range of 18 to 38.5%, and 91.7% to 99.1% of conditions rotate counter-clockwise in each array. In ses-20090910 the values are 15.4% (PMd) and 21.6% (M1), against 26% and 24% for the paper's N-9-10-2010. These are the dandiset's per-electrode units, not the paper's sorted isolations, so the fractions of variance compare different populations.

**Result: reproduced.** The first plane rotates faster for M1 than for PMd in J-array (1.82 against 0.83 Hz) and N-array (1.54 against 1.02 Hz), and in ses-20090910 (1.70 against 0.89 Hz), under every variant shown. The differences are larger than the paper's "slightly": in J-array the M1 frequency is more than twice the PMd one.

## fig6-velocity-model and fig6-complex-kinematic-model

> A simulated population tuned to hand velocity and speed, with preparatory activity tuned to the
> endpoint, shows no robust or consistent rotations; M_skew fits it poorly and its angle
> distribution peaks slightly above zero.
>
> A simulated population tuned to hand position, velocity, acceleration and jerk produces
> multiphasic responses but no consistent rotations.

One simulated unit per recorded unit, from each session's condition-mean hand kinematics, with
spikes drawn per trial at the recorded trial counts (`reach.simulate_population`; the choices the
Methods leave open are listed in its docstring). A lead of 0 ms is a variant.

In [ ]:
rng_models = np.random.default_rng(7016)
rows, sims = [], {}
for name in ("J-array", "N-array"):
    p = prep[name]
    for kind in ("velocity", "complex"):
        for lead in (100, 0):
            sim = reach.simulate_population(kind, kin[name], p["rates"], p["n"], rng_models, lead_ms=lead)
            r = reach.jpca(sim)
            sm = reach.summary(r)
            sims[(name, kind, lead)] = r
            rows.append({"session": name, "model": kind, "lead ms": lead, **sm,
                         "sim rate range": f"{sim.min():.1f} to {sim.max():.1f}"})
    rows.append({"session": name, "model": "neural data", "lead ms": None, **main.loc[name].to_dict(), "sim rate range": ""})
models = pd.DataFrame(rows)
models["angle_peak / pi"] = models.angle_peak / np.pi
print(models[["session", "model", "lead ms", "r2_ratio_6d", "r2_ratio_plane", "angle_peak / pi", "frac_ccw", "plane_var", "freq_hz", "sim rate range"]].round(3).to_string(index=False))
fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
for ax, kind in zip(axes, ("velocity", "complex")):
    proj = sims[("J-array", kind, 100)]["planes"][0]
    for c in range(proj.shape[0]):
        ax.plot(proj[c, :, 0], proj[c, :, 1], lw=0.6)
    ax.set_title(f"J-array, {kind} model")
    print(f"{kind}: jPC1 {proj[..., 0].min():.3f} to {proj[..., 0].max():.3f}, jPC2 {proj[..., 1].min():.3f} to {proj[..., 1].max():.3f}")
plt.tight_layout(); plt.show()

In [ ]:
# What drives the models' angle peak: the preparatory envelope, the lead and the seed. The main
# models above decay their preparatory tuning with a 100 ms time constant from movement onset;
# here it is held on to the end of the window (step) or left out, and the main model is drawn
# with four more seeds.
syn = np.exp(-np.arange(0, 200, 10) / 100.0)[None, :, None] * np.random.default_rng(7020).normal(size=(108, 1, 2))
print(f"Helper check: angle peak of a pure decay toward the origin {reach.angle_peak(reach.state_angles(syn)) / np.pi:.3f} pi\n")
rng_var = np.random.default_rng(7017)
rows, angles = [], {}
for name in ("J-array", "N-array"):
    p = prep[name]
    for kind in ("velocity", "complex"):
        runs = [("decay 100 ms", {}, lead, s) for lead in (100, 0) for s in range(1, 5)]
        runs += [(env, kw, lead, 0) for env, kw in (("step", {"prep_decay_ms": None}),
                                                    ("no preparatory tuning", {"prep_onset_ms": reach.WINDOW_MS[1]}))
                 for lead in (100, 0)]
        for env, kw, lead, seed in runs:
            r = reach.jpca(reach.simulate_population(kind, kin[name], p["rates"], p["n"], rng_var, lead_ms=lead, **kw))
            sm = reach.summary(r)
            angles[(name, kind, env, lead, seed)] = reach.state_angles(r["planes"][0])
            rows.append({"session": name, "model": kind, "preparatory envelope": env, "lead ms": lead, "seed": seed,
                         "angle_peak / pi": sm["angle_peak"] / np.pi, "angle_median / pi": sm["angle_median"] / np.pi,
                         "r2_ratio_6d": sm["r2_ratio_6d"], "frac_ccw": sm["frac_ccw"]})
mvar = pd.DataFrame(rows)
print(mvar.round(3).to_string(index=False))
print("\nFraction of runs whose angle peak lies within pi/4 of zero, by model, envelope and lead:")
print(mvar.assign(below=mvar["angle_peak / pi"].abs() < 0.25).groupby(["model", "preparatory envelope", "lead ms", "session"]).below.mean().unstack().to_string())
fig, axes = plt.subplots(2, 2, figsize=(12, 6), sharex=True)
centres = np.linspace(-1, 1, 37)[:-1] + 1 / 36
for (i, name), (j, kind) in ((a, b) for a in enumerate(("J-array", "N-array")) for b in enumerate(("velocity", "complex"))):
    ax = axes[i, j]
    for env, colour in (("decay 100 ms", "tab:blue"), ("step", "tab:orange"), ("no preparatory tuning", "tab:green")):
        for lead, ls in ((100, "-"), (0, ":")):
            h, _ = np.histogram(angles[(name, kind, env, lead, 1 if env == "decay 100 ms" else 0)], bins=36, range=(-np.pi, np.pi))
            ax.plot(centres, h, color=colour, ls=ls, label=f"{env}, lead {lead} ms")
    ax.set_title(f"{name}, {kind} model"); ax.set_ylabel("count")
axes[1, 0].set_xlabel("angle between x and dx/dt (units of pi)"); axes[1, 1].set_xlabel("angle between x and dx/dt (units of pi)")
axes[0, 0].legend(fontsize=7); plt.tight_layout(); plt.show()

**Result: not reproduced.** The simulated velocity populations fit M_skew poorly, with 6-D ratios of 0.361 (J-array) and 0.255 (N-array) against 0.905 and 0.845 for the data, and fewer of their conditions rotate counter-clockwise (0.630 and 0.778). Their angle distributions peak at 0.806 pi and 0.361 pi, not below pi/4 as the criterion asks, so the criterion fails in both datasets. The angle peak depends on choices the Methods leave open. A pure decay toward the origin peaks near pi (0.972 pi), so the decaying preparatory tuning pushes the peak up: with the lead of 100 ms the peak stays between 0.36 and 0.86 pi over five seeds and under every envelope, and with no lead it lies between 0.19 and 0.42 pi with preparatory tuning (decaying or held). Only with no preparatory tuning and no lead does it peak near zero in both datasets (0.028 pi), as the paper reports for its velocity model. The poor fit of M_skew holds under every variant (6-D ratio 0.08 to 0.55).


**Result: not reproduced.** The complex kinematic populations fit M_skew poorly, with 6-D ratios of 0.082 (J-array) and 0.132 (N-array), 0.289 and 0.207 with no lead, and 0.565 and 0.528 of their conditions rotate counter-clockwise. Their angle peaks are at 0.861 pi and 0.750 pi (0.806 and 0.917 pi with no lead), not below pi/4, so the criterion fails in both datasets. Under the other envelopes, leads and seeds the angle distribution is broad with its peak between 0.19 and 0.97 pi or near -pi; within pi/4 of zero it peaks in J-array only with no preparatory tuning and no lead (-0.083 pi), and in N-array only in one of five seeds with no lead (0.194 pi). The fit of M_skew separates the model from the data as the paper reports (6-D ratio 0.06 to 0.61 across the variants); under none of the choices tried here does the angle distribution peak near zero in both datasets, as in Figure 6a.


## Blocked claims

These claims rest on data the dandiset does not hold, and that the paper did not deposit elsewhere.

- fig1-leech-and-walking: the leech imaging (made by K. Briggman and W. Kristan) and the walking
  monkey's wireless recording are not in the dandiset.
- fig2-single-neuron-responses and fig3abcd-single-electrode-datasets: the 469 single-electrode
  isolations of monkeys A, B, J (J1 to J4) and N are not in the dandiset, which holds only array
  sessions.
- fig4-emg-no-rotation and fig5-generator-model: EMG was not recorded for the array datasets
  (Methods), and the EMG of the single-electrode datasets is not in the dandiset.
- fig6cd-speed-independent-frequency: the speed task of monkeys A and B is not in the dandiset.

In [ ]:
print(f"Total run time {(time.time() - started) / 60:.1f} min")

## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning
and the differences from the paper's method.

| Claim | Outcome |
|---|---|
| array-dataset-composition | partially reproduced |
| fig3ef-same-direction | reproduced |
| fig3-independent-of-curvature | partially reproduced |
| fig3-phase-from-prep-state | partially reproduced |
| var-captured-jpc-plane | reproduced |
| pc23-variance | partially reproduced |
| suppfig4-j-array-planes | not reproduced |
| suppfig4-planes-range | partially reproduced |
| fig6a-angle-peak | reproduced |
| fig6b-skew-fit | reproduced |
| rotation-frequency | partially reproduced |
| suppfig3-shuffle-controls | not reproduced |
| suppfig5-areas-variance | partially reproduced |
| suppfig5-m1-faster | reproduced |
| fig6-velocity-model | not reproduced |
| fig6-complex-kinematic-model | not reproduced |
| fig1-leech-and-walking, fig2-single-neuron-responses, fig3abcd-single-electrode-datasets, fig4-emg-no-rotation, fig5-generator-model, fig6cd-speed-independent-frequency | blocked: data missing |

**How the analysis differs from the paper's.** The units are the dandiset's 192 per session, one
per electrode, where the paper used 146 and 218 sorted isolations. The rates are not PC-smoothed,
and the Gaussian is 20 ms (24 ms is a variant). The jPCA window is -50 to +150 ms from movement
onset for every dataset, the codepack example's, where the paper chose a start per dataset. The
parameters the Methods leave open come from the Python port of the jPCA codepack, as the lab's
own download page could not be fetched. The shuffle controls use one dividing time per dataset
relative to movement onset. The kinematic models are simulated here from the dandiset's hand
kinematics, with a lead, a preparatory time course and modulation depths chosen here. The paper's
averages over nine datasets are compared with the two array datasets the dandiset holds.

**What was found about the dandiset.**

- It holds the paper's two array datasets and eight further array sessions the paper does not
  describe. The seven single-electrode datasets, the EMG, the leech and the walking-monkey data are
  not in it, nor are the sorted isolations of the array datasets: the Units table has one unit per
  electrode and no description of what a unit is.
- In nine of the ten files the first entry of `units/spike_times_index` wrapped at 65536, so unit 0
  is cut short and its remaining spikes open unit 1.
- In five Nitschke files, N-array among them, the clock restarts at recording segments after the
  first, so trials overlap in time and spike times are not sorted; spikes can be placed in trials
  only by their storage order.
- Hand position is stated in metres but is in mm, and is offset in y from the screen coordinates
  by 35 to 53 mm, which the files do not describe.
- The conversion code dates every session of its first conversion in 2009. The file
  `ses-20090910` has the 108 conditions and reached targets of N-array (2010), and is consistent
  with the N-9-10-2010 dataset of Supplementary Figure 5, but its date cannot confirm this.